# Retraining from scratch

We run these models separately from the main experiment so that, when a new unlearned model is made, these checkpoints can be quickly pulled, measured, and then set aside again.

### Imports

In [1]:
import sys
print(sys.version)

3.13.13 | packaged by conda-forge | (main, Apr  8 2026, 02:00:33) [GCC 14.3.0]


In [2]:
import os
import json

In [3]:
%ls

data/                              __pycache__/
evaluation/                        README.md
master_auditor.ipynb               results/
master_hugging_face.ipynb          results_to_replicate.txt
master_hyperparams.py              trainer/
master_pretraining.ipynb           unlearn/
master_retrain_from_scratch.ipynb  visualize_pretraining_results.ipynb
master_unlearning.ipynb            visualize_results.ipynb
models/                            wandb/
_old/


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
# import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


/cs/student/project_msc/2025/ml/jmoncus/tools/miniforge3/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


### Set configs for the pretraining

In [ ]:

from master_hyperparams import hyperparams
device = "cuda" if torch.cuda.is_available() else "cpu"


# ------- MAIN THINGS TO EDIT FOR THIS RUN ------- #
description = "Testing remote SSH into uni machines"
dataset = "CIFAR10"
model_class = "ResNet"
unlearning_type = "random_uniform"
# ------------------------------------------------- #

hp = hyperparams[dataset]
model_hp = hp[model_class]

retrain_config = {

    "description": description,
    
    "device": device,
    "model_class": model_class,
    "unlearning_type": unlearning_type,
    "data": {
        "dataset": dataset,
        "num_classes": hp["num_classes"],
        "batch_size": 1024,  # larger batch for faster pretraining
        "num_workers": hp["num_workers"],
        "item_to_unlearn": hp["items_to_unlearn"][unlearning_type]
        },

    "training": {
        "num_epochs": model_hp["training"]["num_epochs"],
        "num_runs": 3,
        "learning_rate": model_hp["training"]["learning_rate"],
        "weight_decay": model_hp["training"]["weight_decay"],
        "batch_print_freq": 12,
        },
}


### Protocol for several runs

In [ ]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /cs/student/msc/ml/2025/jmoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import glob
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen_lr_annealing
from data.dataloaders import load_dataloaders_for_experiment
import json
from data.utils import setup_seed
import time
from data.utils import split_forget_retain, split_random

def run_retrain_from_scratch(config, checkpoint_folder):
    
    print("="*70)
    print("="*19 + "  " + f'RUNNING RETRAINING FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "="*19)
    print("="*70 + "\n")

    setup_seed(config["GRAND_SEED"])

    # create a subfolder for saving model checkpoints for this retraining
    print(f'All models will be of class {config["model_class"]}.\n')
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config['GRAND_SEED']}", "retrain_from_scratch")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)


    # Save the config for this retraining to the main checkpoints folder
    with open(os.path.join(checkpoint_subfolder, "retrain_from_scratch_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # ... decide what we're unlearning
    item_to_unlearn = config["data"]["item_to_unlearn"]

    # ... announce what we're unlearning
    retrain_name = f"{config['data']['dataset']}_{config['model_class']}_{config['training']['num_epochs']}_epochs_{config['unlearning_type']}_{item_to_unlearn}"
    print("-"*15 + "    " + retrain_name + "\n")
    

    # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
    
    # class_param = item_to_unlearn if config['unlearning_type'] == "class" else None
    # percent_param = item_to_unlearn if config['unlearning_type'] == "percent" else None

    # ----------------------------------------------------------------------------------- #
    # ----------------------------- DEFINE UNLEARNING LOADERS --------------------------- #
    # ----------------------------------------------------------------------------------- #

    # ...  ------------- get some unlearning data for this config ------------------- #
    # ... the dataSET is fixed across runs, and the randomness within runs is handled by simply shuffling the data loader. There is no need to actually apply the micro-seed

    # only need `train`
    marked_train_loader, _, _ = load_dataloaders_for_experiment(
        name = config["data"]["dataset"],
        replace_type=config["unlearning_type"], 
        value_to_replace=item_to_unlearn, 
        batch_size=config["data"]["batch_size"], 
        num_workers=config["data"]["num_workers"], 
        seed = config["GRAND_SEED"], 
        only_mark=True,
        val=False
        )
    # we make sure forget and retain sets are shuffled, to allow randomness across runs
    # only need `retain`
    print("Training - forget vs retain split:")
    _, retain_loader = split_forget_retain(marked_train_loader, batch_size=config["data"]["batch_size"], shuffle = True, num_workers=config["data"]["num_workers"])
    

    
    # ... and do a bunch of runs, where ...
    for i in range(1,  config["training"]["num_runs"]+1):

    
        # ----------------------------------------------------------------------------------- #
        # ----------------------------- RETRAIN FROM SCRATCH -------------------------------- #
        # ----------------------------------------------------------------------------------- #

        run_seed = config["GRAND_SEED"] * 1000 + i
        setup_seed(run_seed)
        
            # ... open new wandb session per run
        wandb.init(
            project="Verifying-Unlearning-2026",
            name=f"{run_seed}_retrain_{retrain_name}",
            config=config,
            reinit= "finish_previous"
            )
        
        print(f" ----- Retraining from scratch for run {i}, {retrain_name} ----- \n")
            
        # ... init a fresh model, opt, criterion, and scheduler for this run_seed
        empty_model = init_model(model_class = config["model_class"], num_classes = config['data']["num_classes"]).to(config["device"])
        opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            opt, 
            T_max=config["training"]["num_epochs"], 
            eta_min=1e-6
            )

        # ... do the training
        retrain_name = f"retrain_run_{i}_{retrain_name}"
        retrain_checkpoint_path = os.path.join(checkpoint_subfolder, f"{retrain_name}.pth")
        start = time.time() # EVENTUALLY NEEDS TO BE MEASURED SOME OTHER WAY
        retrained_model, opt, scheduler, retrain_retain_loss, retrain_retain_acc, retrain_retain_entr, retrain_retain_m_entr = training_regimen_lr_annealing(
            empty_model, 
            retain_loader,
            opt, 
            criterion, 
            scheduler, 
            device = config["device"], 
            num_epochs=config["training"]["num_epochs"], 
            model_path = retrain_checkpoint_path,
            print_freq = config["training"]["batch_print_freq"],
            w_and_b = True
            )
        end = time.time()
        wandb.log({"run time efficiency": end - start})
        

        # closes retrain wandb session
        wandb.finish()


    print("-"*75)
    print("-"*19 + "  " + f'FINISHED RETRAIN FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "-"*19)
    print("-"*75 + "\n")
    

/cs/student/project_msc/2025/ml/jmoncus/virtual-envs/vu2026/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Check metrics on unlearned models

In [ ]:
# MAKE A RANDOM SEED
retrain_config["GRAND_SEED"] = 5000
# DO EXP
run_retrain_from_scratch(config = retrain_config, checkpoint_folder="models/model_checkpoints")

===================  RUNNING RETRAINING FROM SCRATCH, SEED 6  ===================

setup random seed = 6
All models will be of class ResNet.

models/model_checkpoints/seed_6/retrain_from_scratch doesn't exist - creating it...

---------------    CIFAR10_ResNet_100_epochs_random_uniform_0.1

Replacing 5000 samples total (10.0% across all 10 classes)
Replacing indeces: [21878 32016  3347  4477 23602 26505  5956 28833 19347 37561] ...
========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Replace type = random_uniform, value to replace = 0.1
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


Training - forget vs retain split:
Forget set: 5000 items
Retain set: 45000 items


setup random seed = 6001


 ----- Retraining from scratch for run 1, CIFAR10_ResNet_100_epochs_random_uniform_0.1 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.7796 (2.0832)	Accuracy 35.840 (26.139)	Entropy 1.6863 (1.8316)	M-Entropy 1.7038 (2.0521)	Time 3.85
Epoch: [1][23/44]	Loss 1.5514 (1.8347)	Accuracy 43.945 (33.549)	Entropy 1.5139 (1.7053)	M-Entropy 1.4880 (1.7800)	Time 2.56
Epoch: [1][35/44]	Loss 1.3982 (1.7069)	Accuracy 47.461 (37.823)	Entropy 1.4021 (1.6275)	M-Entropy 1.3559 (1.6490)	Time 2.57
train_accuracy (epoch) 40.022
Epoch 1 | LR: 1.0e-03 | RAM: 2.04GB | VRAM: 7.08GB | Weight Norm: 111.954
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.2055 (1.2790)	Accuracy 57.031 (53.312)	Entropy 1.2744 (1.3006)	M-Entropy 1.1543 (1.2404)	Time 3.35
Epoch: [2][23/44]	Loss 1.2033 (1.2447)	Accuracy 57.031 (54.708)	Entropy 1.1338 (1.2614)	M-Entropy 1.2265 (1.2120)	Time 2.57
Epoch: [2][35/44]	Loss 1.0705 (1.2097)	Accuracy 63.867 (56.082)	Entropy 1.1147 (1.2212)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▁▄▇▇▇▇▇▇███████████████████▅▅▅▅▅▅▅▅▅▅▅▅▅
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
learning_rate,██████▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
run time efficiency,▁
time (batch),▁▁▁▁▇▁▁▂▇█▇▁▇▁▁██▇▇▂▁▂▇▁▇▁▇█▁▁▂▁▂▇▂▂▂▂▂▂
train_acc (batch),▁▄▆▆▆▆▆▆▇▇▇▇▇▇██████████████████████████
train_acc (full),▁▂▃▅▅▆▆▆▇▇▇▇▇▇▇▇▇███████████████████████
train_entropy (batch),██▅▆▆▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▇▆▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


setup random seed = 6002


...

setup random seed = 6003


 ----- Retraining from scratch for run 3, retrain_run_2_retrain_run_1_CIFAR10_ResNet_100_epochs_random_uniform_0.1 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.6859 (2.0817)	Accuracy 38.379 (24.552)	Entropy 1.7529 (1.8784)	M-Entropy 1.5668 (2.0144)	Time 3.16
Epoch: [1][23/44]	Loss 1.5794 (1.8490)	Accuracy 40.723 (32.430)	Entropy 1.5064 (1.7494)	M-Entropy 1.5304 (1.7713)	Time 2.44
Epoch: [1][35/44]	Loss 1.4265 (1.7240)	Accuracy 50.000 (36.825)	Entropy 1.4014 (1.6614)	M-Entropy 1.3820 (1.6503)	Time 2.44
train_accuracy (epoch) 38.860
Epoch 1 | LR: 1.0e-03 | RAM: 2.08GB | VRAM: 7.09GB | Weight Norm: 111.884
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.2777 (1.3199)	Accuracy 57.227 (51.571)	Entropy 1.3044 (1.3102)	M-Entropy 1.2430 (1.2869)	Time 3.16
Epoch: [2][23/44]	Loss 1.2208 (1.2823)	Accuracy 56.641 (53.284)	Entropy 1.2238 (1.2841)	M-Entropy 1.1834 (1.2500)	Time 2.45
Epoch: [2][35/44]	Loss 1.1471 (1.2446)	Accuracy 59.863 (54.

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▁███████████████████████████████████████
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
learning_rate,███████████▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁
run time efficiency,▁
time (batch),▁▇▁▇▁▁▁▇▁▇▁▇▁▁▁▂▂▁▁▇▂▂▇█▂▂▇▂▇▂▇▂▂▂▂▁▁▁▁▁
train_acc (batch),▁▄▅▅▆▆▇▇▇▇▇▇▇▇██████████████████████████
train_acc (full),▁▃▄▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████████████████
train_entropy (batch),█▅▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▆▆▅▅▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


---------------------------------------------------------------------------
-------------------  FINISHED RETRAIN FROM SCRATCH, SEED 6  -------------------
---------------------------------------------------------------------------

